# Predicting Formula 1 Mechanical DNFs (2014–2025)

**DSA 210 Term Project — Milestone 3 (ML phase)**

### What this notebook does, in one paragraph

We have one row per driver per race, for every Formula 1 race from 2014 to
2025 — about 4,860 rows in total, of which roughly 4 % end in a mechanical
DNF (engine, gearbox, electrical, hydraulics, overheating, power-unit, or
generic mechanical failure). We ask four questions about *what predicts
those failures*: altitude (H1), street-vs-permanent circuit (H2),
seasonal fatigue (H3), and how worn the engine is at the time of the race
(H4). For each question we run a simple statistical test, plot the data,
and decide whether the hypothesis is supported. We then train two
machine-learning models (logistic regression and random forest) on the
same features to see how predictable a DNF actually is.

### What's different from the earlier `main.py`

This is a notebook, not a script, so every intermediate result is
visible. Two methodological issues from the earlier analysis are also
fixed here:

1. **H1 (altitude)** is now tested at the *circuit* level (one row per
   circuit), not the driver-race level. Altitude is shared by every
   driver in a given race, so treating those rows as independent
   inflated the apparent precision of the earlier test.
2. **H4 (engine cycle)** previously assumed a fixed 7-race engine
   lifespan. There is no such rule in F1; the FIA combustion-engine
   quota varies by season (5 in 2014, 4 in 2015–17, 3 in 2018–22, 4
   in 2023–25). We rebuild the cycle-position feature from those
   actual quotas.

The dataset itself is loaded from the CSV produced by `main.py`, so the
notebook does not need to call the FastF1 API.

## 1. Setup and imports

Standard scientific-Python stack. The fixed `RANDOM_STATE` makes the
train/test split and the random forest reproducible, so re-running the
notebook gives identical numbers.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports OK")

## 2. Load the prepared dataset

The CSV came out of the data-collection phase: every race result from
2014–2025 was pulled with FastF1, enriched with the circuit's altitude
and circuit type, and then labelled. Each row is one driver in one race.
The label `is_mechanical_dnf` is `1` if the FIA finishing status names a
mechanical cause — engine, gearbox, electrical, hydraulics, overheating,
power unit, or just "mechanical". Driver-error retirements (Accident,
Collision) were stripped out earlier because they do not measure car
reliability.

In [ ]:
DATA_PATH = "f1_term_project_data.csv"
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,} | Drivers: {df['Driver'].nunique()} | "
      f"Circuits: {df['Circuit'].nunique()} | Years: {df['Year'].min()}–{df['Year'].max()}")
df.head()

In [ ]:
# Sanity checks
print("Columns:", list(df.columns))
print("\nMissing values:")
print(df.isna().sum())
print(f"\nOverall mechanical-DNF rate: {df['is_mechanical_dnf'].mean():.3%}")

## 3. Quick exploratory look

Three quick charts to see what we're working with before any tests:
the DNF rate per season, the DNF rate split by circuit type, and the
distribution of altitudes across the calendar. The headline number to
remember is that mechanical DNFs are rare — only about 4 % of all
driver-race rows. That class imbalance comes back to bite us in the ML
section, where plain accuracy stops being a useful metric.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# DNF rate per year
yearly = df.groupby("Year")["is_mechanical_dnf"].mean()
axes[0].bar(yearly.index, yearly.values, color="steelblue")
axes[0].set_title("Mechanical-DNF rate by season")
axes[0].set_ylabel("DNF rate")
axes[0].set_xlabel("Year")

# DNF rate per circuit type
type_rate = df.groupby("Type")["is_mechanical_dnf"].mean()
axes[1].bar(type_rate.index, type_rate.values, color=["coral", "seagreen"])
axes[1].set_title("DNF rate by circuit type")
axes[1].set_ylabel("DNF rate")

# Altitude distribution
axes[2].hist(df["Altitude"], bins=30, color="slategrey")
axes[2].set_title("Altitude distribution (per driver-race row)")
axes[2].set_xlabel("Altitude (m)")

plt.tight_layout()
plt.show()

## 4. Hypothesis 1 — Altitude effect (corrected)

> **H1**: Mechanical DNFs happen more often at high-altitude circuits.
> Thinner air should mean hotter engines and more cooling problems.

### 4.1 Why we changed how we test this

The earlier version compared the altitude of "DNF rows" against the
altitude of "finished rows" using a t-test. That sounds reasonable but
it is wrong, because **altitude is a property of the circuit, not of
each driver**. Every car in the Mexico GP shares the same 2,285 m
altitude — they aren't 20 independent measurements of "altitude
during a DNF". With about 20 drivers per race and 12 seasons, each
circuit's altitude is duplicated up to ~240 times in the data. A
t-test that takes those duplicates seriously over-counts our sample
and reports a precision we don't actually have.

### 4.2 The corrected approach

The right unit of analysis for H1 is the **circuit**. We collapse the
table to one row per circuit, compute the actual mechanical-DNF rate
seen at that circuit across all years, and then ask whether that rate
goes up with altitude. We end up with about 30 circuits, all of them
genuinely independent. We run two correlation tests: Pearson (linear)
and Spearman (rank-based, robust to outliers like Mexico City). We
also keep the old per-row test alongside, so the inflation is
explicit.

In [ ]:
# --- (a) Naive per-row test from the original script (kept for comparison) ---
mech = df.loc[df["is_mechanical_dnf"] == 1, "Altitude"]
fin  = df.loc[df["is_mechanical_dnf"] == 0, "Altitude"]
t_naive, p_naive = stats.ttest_ind(mech, fin, equal_var=False)
print("Naive Welch t-test on driver-race rows (INCORRECT — pseudo-replicated):")
print(f"   t = {t_naive:.3f},   p = {p_naive:.4g}")
print(f"   n_DNF = {len(mech):,},   n_finished = {len(fin):,}")
print(f"   mean altitude (DNF):     {mech.mean():7.1f} m")
print(f"   mean altitude (finish):  {fin.mean():7.1f} m")

In [ ]:
# --- (b) Corrected: aggregate to circuit level ---
circuit_stats = (
    df.groupby("Circuit")
      .agg(altitude=("Altitude", "first"),
           type_=("Type", "first"),
           races=("Driver", "count"),
           dnf_rate=("is_mechanical_dnf", "mean"))
      .reset_index()
)
# only circuits with enough exposure to give a stable rate
circuit_stats = circuit_stats[circuit_stats["races"] >= 20].copy()
print(f"Circuits used: {len(circuit_stats)} (rows >= 20 driver-races each)")
circuit_stats.sort_values("altitude", ascending=False).head(10)

In [ ]:
# Pearson and Spearman on the circuit-level data
pearson_r,  pearson_p  = stats.pearsonr(circuit_stats["altitude"], circuit_stats["dnf_rate"])
spearman_r, spearman_p = stats.spearmanr(circuit_stats["altitude"], circuit_stats["dnf_rate"])

print("Circuit-level correlation between altitude and DNF rate:")
print(f"   Pearson  r = {pearson_r:+.3f}   p = {pearson_p:.4g}")
print(f"   Spearman r = {spearman_r:+.3f}   p = {spearman_p:.4g}")

In [ ]:
# Visualise the corrected analysis
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(circuit_stats["altitude"], circuit_stats["dnf_rate"],
           s=circuit_stats["races"]/2, alpha=0.7, color="firebrick", edgecolor="k")
for _, row in circuit_stats.iterrows():
    if row["altitude"] > 400 or row["dnf_rate"] > 0.07:
        ax.annotate(row["Circuit"],
                    (row["altitude"], row["dnf_rate"]),
                    fontsize=8, alpha=0.8)
# best fit line
m, b = np.polyfit(circuit_stats["altitude"], circuit_stats["dnf_rate"], 1)
xs = np.linspace(circuit_stats["altitude"].min(), circuit_stats["altitude"].max(), 100)
ax.plot(xs, m*xs + b, "--", color="grey",
        label=f"OLS fit (slope={m:.2e})")
ax.set_xlabel("Altitude (m)")
ax.set_ylabel("Mechanical-DNF rate at the circuit")
ax.set_title("H1 corrected: per-circuit DNF rate vs altitude\n"
             f"Pearson r = {pearson_r:+.2f}, p = {pearson_p:.3f}")
ax.legend()
plt.tight_layout()
plt.savefig("graph1_altitude_effect_corrected.png", dpi=130)
plt.show()

### 4.3 What we found — H1 is not supported

**Short version:** altitude does not predict mechanical DNFs in this
dataset. The rank-based test even leans the other way.

The numbers from the three tests above:

| Test                                       | Statistic        | p-value | Verdict             |
|--------------------------------------------|------------------|---------|---------------------|
| Naive per-row Welch t-test (incorrect)     | t = 0.27         | 0.79    | no effect           |
| Circuit-level Pearson correlation          | r = +0.05        | 0.77    | essentially zero    |
| Circuit-level Spearman (rank correlation)  | r = −0.11        | 0.53    | wrong sign for H1   |

The descriptive picture is even clearer than the test statistics:

* The single worst circuit for reliability is **Spa** (mid-altitude,
  418 m) at a **13.75 %** DNF rate.
* The highest-altitude circuit by a huge margin is **Mexico City**
  (2,285 m), and it sits at only **4.7 %** — barely above the
  ~3.7 % overall average.

If the high-altitude story were correct, Mexico City should be the
worst circuit. It isn't even close. The very small positive Pearson
slope is essentially Mexico City alone tugging the regression line
upward; the rank-based Spearman, which doesn't care about that one
extreme x-value, actually goes negative.

The methodology correction still matters in principle. If the per-row
t-test had happened to come out significant, that significance would
have been a statistical artefact — the same circuit altitude is
duplicated up to ~240 times in the row-level layout, so the t-test
"thinks" it has 4,800 independent altitude measurements when in
reality it has about 30. Aggregating to one row per circuit removes
that artefact.

A stricter future analysis would use a mixed-effects logistic
regression with circuit as a random intercept, which would let us
also hold constructor and year fixed.

## 5. Hypothesis 2 — Circuit type (Street vs Permanent)

> **H2**: Street circuits — concrete walls, kerbs, less run-off,
> bumpier surfaces — should be harder on the car and produce more
> mechanical DNFs than permanent racetracks.

We split every driver-race into two groups (Permanent vs Street),
count finishes and DNFs in each group, and run a **chi-square test
of independence** to ask "is the DNF rate actually different between
the two types, or are we seeing noise?". Both variables here are
categorical and each driver-race counts once, so chi-square is the
natural test.

In [ ]:
contingency = pd.crosstab(df["Type"], df["is_mechanical_dnf"])
print("Contingency table:")
print(contingency)

chi2, p_type, dof, expected = stats.chi2_contingency(contingency)
print(f"\nchi2 = {chi2:.3f}   p = {p_type:.4g}   dof = {dof}")

print("\nDNF rate by circuit type:")
print(df.groupby("Type")["is_mechanical_dnf"].mean().round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=df, x="Type", y="is_mechanical_dnf",
            estimator=np.mean, palette="viridis", ax=ax)
ax.set_ylabel("Mechanical-DNF probability")
ax.set_title("H2: DNF rate by circuit type")
plt.tight_layout()
plt.savefig("graph2_circuit_type.png", dpi=130)
plt.show()

### 5.1 What we found — H2 is not supported

The DNF rates barely differ:

| Circuit type | Driver-race rows | DNFs | DNF rate |
|--------------|------------------|------|----------|
| Permanent    | 3,777            | 138  | 3.65 %   |
| Street       | 1,083            | 44   | 4.06 %   |

The chi-square statistic is **0.39** with 1 degree of freedom, giving
**p ≈ 0.53** — nowhere near the 0.05 threshold. With ~4,800 driver-
races we had ample statistical power to detect even a modest
difference, so this is not "we couldn't tell because the sample is
small". The street-vs-permanent gap really is too small to call.

In plain words: **a car running at Monaco, Singapore or Baku is
about as likely to suffer a mechanical DNF as a car running at
Silverstone, Spa or Monza.** Whatever causes Spa's terrible
reliability number, it is not the "street circuit" label.

## 6. Hypothesis 3 — Seasonal fatigue (Round number)

> **H3**: Cars get tireder as the season goes on. Engines, gearboxes
> and electronics accumulate kilometres race-by-race, so we should see
> *more* DNFs late in the season than early in it.

We compare the "round number" (1 = season opener, ~22 = season finale)
between rows that ended in a mechanical DNF and rows that finished.
If H3 is right, the average round number for DNF rows should be
*higher* than for finished rows.

In [ ]:
# Welch t-test on Round number (kept from the original analysis but
# also visualised by season for context)
rd_dnf = df.loc[df["is_mechanical_dnf"] == 1, "Round"]
rd_fin = df.loc[df["is_mechanical_dnf"] == 0, "Round"]
t_round, p_round = stats.ttest_ind(rd_dnf, rd_fin, equal_var=False)
print(f"Welch t on Round: t = {t_round:.3f}, p = {p_round:.4g}")
print(f"Mean Round, DNF rows:    {rd_dnf.mean():.2f}")
print(f"Mean Round, finish rows: {rd_fin.mean():.2f}")

In [ ]:
# DNF rate as a function of round number
rate_by_round = df.groupby("Round")["is_mechanical_dnf"].agg(["mean", "count"]).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rate_by_round["Round"], rate_by_round["mean"],
        marker="o", color="firebrick")
ax.set_title("H3: mechanical-DNF rate vs round number (all seasons pooled)")
ax.set_xlabel("Round")
ax.set_ylabel("DNF rate")
plt.tight_layout()
plt.savefig("graph3_seasonal_fatigue.png", dpi=130)
plt.show()

### 6.1 What we found — H3 is rejected (and the data points the other way)

The Welch t-test gives **t = -2.18, p ≈ 0.029** — statistically
significant, but with the **wrong sign**:

| Group           | Mean round number |
|-----------------|-------------------|
| Mechanical DNFs | **10.15**         |
| Finishers       | **11.13**         |

DNF rows come from *earlier* rounds, not later. Splitting the season
into thirds makes the pattern even clearer:

| Round bucket | n     | DNFs | DNF rate  |
|--------------|-------|------|-----------|
| 1 – 6        | 1,387 | 55   | 3.97 %    |
| 7 – 12       | 1,388 | 58   | 4.18 %    |
| 13 – 18      | 1,369 | 52   | 3.80 %    |
| 19 – 24      | 716   | 17   | **2.37 %**|

Reliability *improves* in the second half of the calendar, with the
late races (rounds 19–24) showing only ~2.4 % mechanical DNFs.

The most likely explanation is a **calendar-growth confound**, not
literal seasonal fatigue. The F1 calendar grew from 19 rounds in 2014
to 24 rounds in 2024–25, so rounds 19–24 *only exist in the recent,
more reliable seasons*. We are partly measuring "modern hybrid PUs are
more durable than 2014–17 ones", not "engines are fresh at Bahrain".

So H3 as written is rejected. A cleaner future test would either
normalise round number to a 0–1 fraction of the season, or fit a
year-by-year slope, to disentangle within-season trend from
across-year improvement.

## 7. Hypothesis 4 — Engine-life position (corrected)

> **H4**: An engine becomes more failure-prone as it gets older. So
> the closer we are to the end of an engine's planned lifespan, the
> higher we expect the DNF rate to be.

### 7.1 What was wrong with the previous test

The earlier version of this analysis grouped every race by
`Round % 7` — i.e. assumed each engine is meant to last exactly
7 races, so race 1, 8, 15, … are "fresh engine" and race 7, 14, 21
are "late engine". That 7-race window doesn't come from anywhere in
the rulebook. F1's actual power-unit allocation **changes from season
to season**, and there is no single "every 7 races" cycle that
applies across the hybrid era. Using a fixed mod-7 grouping was an
unjustified modelling choice that made the test look more rigorous
than it was.

### 7.2 The corrected approach — use the actual FIA quota

For each season we look up the **real** FIA limit on combustion
engines per driver, before grid penalties are triggered:

| Seasons      | ICE per driver per season |
|--------------|---------------------------|
| 2014         | 5                         |
| 2015 – 2017  | 4                         |
| 2018 – 2022  | 3                         |
| 2023 – 2025  | 4                         |

Then for each season we compute the *expected* number of races each
engine has to cover as `total_races_in_season / quota`. (Example:
2024 had 24 rounds and a quota of 4, so each engine is expected to
cover 6 races on average.)

For every driver-race row we then compute a **normalised position
in the engine cycle** between 0 and 1:

* 0 means "fresh engine, just installed"
* close to 1 means "near the end of this engine's planned life"

Finally we bin those positions into thirds — *early*, *mid*, *late*
— and compare the DNF rates between thirds. H4 predicts the *late*
third should be highest.

This is still an approximation, because in real life teams sometimes
take a strategic grid penalty and swap engines out of order, and
because some of our DNFs are gearbox/hydraulics/electrical (which the
FIA combustion-engine quota does not govern). So H4 should be read
as **exploratory** — even with the corrected cycle, we cannot directly
observe each engine's true age. The honest fix would be using the
FIA's per-driver PU-installation logs.

In [ ]:
# FIA combustion-engine quota per driver per season (hybrid era)
ENGINE_QUOTA = {
    2014: 5,
    2015: 4, 2016: 4, 2017: 4,
    2018: 3, 2019: 3, 2020: 3, 2021: 3, 2022: 3,
    2023: 4, 2024: 4, 2025: 4,
}

# Season length in actual rounds (from the data itself, so we don't
# hard-code something that disagrees with the CSV).
season_length = df.groupby("Year")["Round"].max().to_dict()

def normalised_engine_pos(row):
    quota = ENGINE_QUOTA.get(int(row["Year"]))
    n_rounds = season_length[int(row["Year"])]
    if quota is None:
        return np.nan
    cycle_len = n_rounds / quota                      # expected races per PU
    pos_in_cycle = ((row["Round"] - 1) % cycle_len)   # 0 .. cycle_len
    return pos_in_cycle / cycle_len                   # 0 .. 1

df["Engine_Cycle_Norm"] = df.apply(normalised_engine_pos, axis=1)
df[["Year", "Round", "Engine_Cycle_Norm"]].head(10)

In [ ]:
# Bin the normalised position into thirds: early / middle / late life
df["Engine_Phase"] = pd.cut(
    df["Engine_Cycle_Norm"],
    bins=[-0.001, 1/3, 2/3, 1.0001],
    labels=["early", "mid", "late"],
)
phase_table = df.groupby("Engine_Phase")["is_mechanical_dnf"].agg(
    ["count", "sum", "mean"]
).rename(columns={"count": "n", "sum": "dnf", "mean": "dnf_rate"})
print(phase_table)

# Welch t-test on the binary DNF outcome between late and early phases
late  = df.loc[df["Engine_Phase"] == "late",  "is_mechanical_dnf"]
early = df.loc[df["Engine_Phase"] == "early", "is_mechanical_dnf"]
t_eng, p_eng = stats.ttest_ind(late, early, equal_var=False)
print(f"\nWelch t (late vs early): t = {t_eng:.3f}, p = {p_eng:.4g}")

In [ ]:
# Visualise: DNF rate vs normalised position-in-PU-cycle
df["pos_bin"] = pd.cut(df["Engine_Cycle_Norm"], bins=10)
bin_rate = (
    df.groupby("pos_bin", observed=True)["is_mechanical_dnf"].mean().reset_index()
)
bin_rate["mid"] = bin_rate["pos_bin"].apply(lambda iv: iv.mid)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(bin_rate["mid"], bin_rate["is_mechanical_dnf"],
        marker="o", color="darkorange")
ax.set_xlabel("Normalised position in PU cycle (0 = fresh, 1 = late)")
ax.set_ylabel("Mechanical-DNF rate")
ax.set_title("H4 corrected: DNF rate across the FIA-quota-adjusted engine cycle")
plt.tight_layout()
plt.savefig("graph4_engine_lifecycle_corrected.png", dpi=130)
plt.show()

### 7.3 What we found — H4 is not supported

The FIA-quota-adjusted cycle gives almost the same DNF rate at all
three life phases:

| Engine phase | n     | DNFs | DNF rate |
|--------------|-------|------|----------|
| early        | 1,972 | 76   | 3.85 %   |
| mid          | 1,586 | 53   | 3.34 %   |
| late         | 1,302 | 53   | 4.07 %   |

Late-vs-early difference: only **0.22 percentage points**. The
Welch t-test gives **t = 0.31, p ≈ 0.76** — not even close to
significant. Looking at the 10-bin breakdown, the rate jumps around
between roughly 2 % and 6 % with no clean "early low → late high"
trend; it looks like noise.

So with the corrected, more defensible cycle definition, H4 is **not
supported** by the data. The "engines fail more at the end of their
life" intuition does *not* show up at the level of mechanical DNFs,
at least not with the cycle proxy we can build from quotas alone. The
caveats in §7.2 still apply — strategic engine rotation and the
inclusion of non-engine failures (gearbox, hydraulics, electrical) in
our label both blur the signal — but the original mod-7 framing made
the result look much more decisive than it deserved.

## 8. Machine-learning phase

The hypothesis tests above tell us about *individual* predictors one
at a time. The next question is: *given everything we know about a
race together — the year, the round number, the altitude, the
circuit type, the engine-cycle position, and which constructor the
car belongs to — how well can we predict whether a particular
driver-race will end in a mechanical DNF?*

We train two classifiers:

* **Logistic Regression** — a simple linear model. Easy to read off
  which features push the predicted DNF probability up or down, and
  how strongly. Good as a reference baseline.
* **Random Forest** — an ensemble of decision trees. Captures
  non-linear interactions automatically and gives a "feature
  importance" ranking.

Important: the target is very imbalanced (only ~4 % of rows are
DNFs). A naive classifier that always predicts "finished" already
scores 96 % accuracy — but it has zero useful predictive power. So
we train both models with `class_weight="balanced"` and report
**ROC-AUC** as the headline metric instead of accuracy.

In [ ]:
# --- Feature engineering for ML ---
ml_df = df.copy()

# One-hot encode circuit type
ml_df = pd.get_dummies(ml_df, columns=["Type"], prefix="Type", drop_first=True)

# Constructor: keep the top-N constructors by row count, group the rest as
# 'Other' to keep the feature space manageable
top_constructors = ml_df["Constructor"].value_counts().head(10).index
ml_df["Constructor_grp"] = ml_df["Constructor"].where(
    ml_df["Constructor"].isin(top_constructors), other="Other"
)
ml_df = pd.get_dummies(ml_df, columns=["Constructor_grp"], prefix="C")

feature_cols = (
    ["Year", "Round", "Altitude", "Engine_Cycle_Norm"]
    + [c for c in ml_df.columns if c.startswith("Type_")]
    + [c for c in ml_df.columns if c.startswith("C_")]
)
ml_df = ml_df.dropna(subset=feature_cols + ["is_mechanical_dnf"])

X = ml_df[feature_cols].astype(float).values
y = ml_df["is_mechanical_dnf"].astype(int).values
print(f"Feature matrix: {X.shape},  positives: {y.sum()} ({y.mean():.2%})")
print("Features:", feature_cols)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape},  Test: {X_test.shape}")
print(f"Train DNF rate: {y_train.mean():.3%},  Test DNF rate: {y_test.mean():.3%}")

In [ ]:
# --- Logistic Regression ---
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

logit = LogisticRegression(class_weight="balanced",
                           max_iter=2000, random_state=RANDOM_STATE)
logit.fit(X_train_s, y_train)

y_pred_logit = logit.predict(X_test_s)
y_prob_logit = logit.predict_proba(X_test_s)[:, 1]

print("Logistic Regression — classification report:")
print(classification_report(y_test, y_pred_logit, digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_logit):.3f}")

In [ ]:
# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest — classification report:")
print(classification_report(y_test, y_pred_rf, digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.3f}")

In [ ]:
# --- Cross-validated AUC for a more stable comparison ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
auc_logit = cross_val_score(logit, scaler.fit_transform(X), y,
                            cv=cv, scoring="roc_auc")
auc_rf    = cross_val_score(rf, X, y, cv=cv, scoring="roc_auc")
print(f"Logit  5-fold ROC-AUC: {auc_logit.mean():.3f} ± {auc_logit.std():.3f}")
print(f"RF     5-fold ROC-AUC: {auc_rf.mean():.3f} ± {auc_rf.std():.3f}")

In [ ]:
# --- ROC curves for both models ---
fpr_l, tpr_l, _ = roc_curve(y_test, y_prob_logit)
fpr_r, tpr_r, _ = roc_curve(y_test, y_prob_rf)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr_l, tpr_l, label=f"Logistic Regression (AUC={roc_auc_score(y_test, y_prob_logit):.3f})")
ax.plot(fpr_r, tpr_r, label=f"Random Forest (AUC={roc_auc_score(y_test, y_prob_rf):.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves on held-out test set")
ax.legend()
plt.tight_layout()
plt.savefig("graph5_roc.png", dpi=130)
plt.show()

In [ ]:
# --- Random Forest feature importance ---
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()
fig, ax = plt.subplots(figsize=(8, max(4, 0.32*len(importance))))
importance.plot(kind="barh", ax=ax, color="teal")
ax.set_title("Random Forest feature importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("graph6_feature_importance.png", dpi=130)
plt.show()

print("\nTop 10 features by importance:")
print(importance.sort_values(ascending=False).head(10).round(4))

In [ ]:
# --- Logistic Regression coefficients (signed effect, on standardised features) ---
logit_coef = pd.Series(logit.coef_[0], index=feature_cols).sort_values()
print("Logistic Regression coefficients (standardised features):")
print(logit_coef.round(3))

### 8.1 How to read the ML results

A few things to keep in mind when looking at the printed metrics:

* **ROC-AUC** is the main number. It ranges from 0.5 (random
  guessing) to 1.0 (perfect prediction). Roughly: 0.6 = "weak but
  real signal", 0.7+ = "useful", 0.8+ = "strong". Realistic AUCs for
  noisy reliability prediction with this feature set are in the
  0.55–0.70 range — F1 failures are mostly driven by component-level
  details we don't have in the data.
* **Precision** = of the rows the model flagged as DNF, what
  fraction really were DNFs?
* **Recall** = of the actual DNFs, what fraction did the model
  catch?
* Because the positive class is tiny (~4 %), accuracy is misleading.
  We trained with `class_weight="balanced"`, which tells the model
  to pay extra attention to the rare DNF class even at the cost of
  false alarms.

The most informative output of this section is the **feature
importance ranking** from the random forest, plus the signed
coefficients from logistic regression. They tell us *which variables
the model leaned on* — and importantly, that ranking is independent
of whether each variable's individual hypothesis test reached
p < 0.05. A feature can be statistically "non-significant" on its
own and still help the model when combined with others.

## 9. Conclusions

The four hypotheses in one table:

| # | Hypothesis                                    | Test                                | Result                                | Verdict       |
|---|-----------------------------------------------|-------------------------------------|---------------------------------------|---------------|
| 1 | DNFs more frequent at high-altitude circuits  | Pearson r at circuit level          | r = +0.05, p = 0.77 (Spearman = -0.11) | **rejected**  |
| 2 | Street circuits cause more DNFs               | chi-square on Type × DNF            | chi² = 0.39, p = 0.53                  | **rejected**  |
| 3 | More DNFs late in the season                  | Welch t on Round                    | t = -2.18, p = 0.029 — but *opposite* sign | **rejected (opposite sign)** |
| 4 | More DNFs late in an engine's life            | Welch t on early-vs-late engine phase | t = 0.31, p = 0.76                     | **rejected**  |

Plain-language summary:

* **Altitude doesn't matter** for mechanical DNFs at the circuit
  level. The worst-reliability circuit is mid-altitude Spa
  (~13.7 %), while the highest-altitude circuit, Mexico City, is
  unremarkable (~4.7 %).
* **Street vs permanent doesn't matter.** 4.06 % vs 3.65 % —
  practically indistinguishable.
* **Seasonal fatigue is the wrong story.** DNF rate actually
  *drops* in the late season, almost certainly because the late
  rounds (19–24) only exist in the recent, more-reliable seasons
  of the hybrid era, not because engines are fresh.
* **Engine-life position doesn't matter** in the way the original
  hypothesis claimed, even after we replaced the made-up 7-race
  cycle with the actual FIA quota.

The single most important predictor for these mechanical DNFs in
the modern era is therefore **none of the four things we
hypothesised**. The strongest signal in the data is just that some
constructors (and some seasons) are simply more reliable than
others — the random-forest feature importance is consistent with
that reading.

For the ML phase: random forest performs comparably to logistic
regression on cross-validated ROC-AUC, but neither model achieves
strong predictive power on this feature set. That is a meaningful
result on its own — it tells us mechanical DNFs are hard to predict
from coarse race-level features and are dominated by component-level
details we don't observe.

## 10. Limitations and future work

* **No direct measure of component age.** We approximate where each
  car is in its engine cycle, but we don't actually know when each
  power unit was fitted. The FIA's per-race technical bulletins
  list PU-installation events explicitly — pulling those would let
  H4 be tested for real, instead of through a quota proxy.
* **No weather data yet.** The original proposal also planned to add
  ambient temperature and rainfall via Open-Meteo. That's a natural
  next addition: thermal stress is a far more direct mechanism for
  reliability than altitude alone.
* **Calendar growth is a confound.** The 2014 season had 19 rounds;
  2024–25 had 24. Any analysis that uses raw round number conflates
  "how late in the season" with "which era" (and the modern era is
  more reliable). Re-expressing round number as a fraction of the
  season, or fitting a year-by-year trend, would separate the two.
* **A single principled model would be cleaner.** Running four
  separate hypothesis tests is simple to explain, but a
  mixed-effects logistic regression with random intercepts for
  circuit and constructor would let us hold all the variables
  fixed at once and read off each effect with the right standard
  errors. We chose the four-tests format here because it maps
  cleanly onto the project's stated hypothesis structure.